**Cell 1 — Mount + extract + split:**

In [25]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

SOURCE_ZIP = '/content/drive/MyDrive/Thesis /Final /archive.zip'
DEST_PATH = '/content/data'
os.makedirs(DEST_PATH, exist_ok=True)
with zipfile.ZipFile(SOURCE_ZIP, 'r') as z:
    z.extractall(DEST_PATH)

DATA_PATH = '/content/data/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone'

!pip install split-folders -q
import splitfolders
splitfolders.ratio(DATA_PATH, output="/content/split_data", seed=42, ratio=(.7, .15, .15))
print("Done.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Copying files: 12446 files [00:15, 801.07 files/s]

Done.


**Cell 2 — Install timm:**

In [26]:
!pip install timm -q

**Cell 3 — PyTorch data loading:**

In [27]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using:", device)

IMG_SIZE = 224
BATCH = 32

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(11),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.92, 1.08)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

train_data = datasets.ImageFolder('/content/split_data/train', transform=train_transform)
val_data = datasets.ImageFolder('/content/split_data/val', transform=eval_transform)
test_data = datasets.ImageFolder('/content/split_data/test', transform=eval_transform)

train_loader = DataLoader(train_data, batch_size=BATCH, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH, shuffle=False)
test_loader = DataLoader(test_data, batch_size=BATCH, shuffle=False)

class_names = train_data.classes
print("Classes:", class_names)

Using: cuda
Classes: ['Cyst', 'Normal', 'Stone', 'Tumor']


**Cell 4 — Sanity check**

In [28]:
print("Train:", len(train_data), "| Val:", len(val_data), "| Test:", len(test_data))

train_files = set(f for f, _ in train_data.samples)
val_files = set(f for f, _ in val_data.samples)
test_files = set(f for f, _ in test_data.samples)

print("Train-Val overlap:", len(train_files & val_files))
print("Train-Test overlap:", len(train_files & test_files))
print("Val-Test overlap:", len(val_files & test_files))

Train: 8710 | Val: 1865 | Test: 1871
Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0


In [21]:
!pip install timm -q

**Cell 5 — Model define:**

In [32]:
model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=len(class_names))
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
criterion = nn.CrossEntropyLoss()

**Cell 6 — Training loop:**

In [33]:
import time

EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0
    start = time.time()
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_acc, train_loss = correct / total, total_loss / total

    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_acc, val_loss = val_correct / val_total, val_loss / val_total

    print(f"Epoch {epoch+1}/{EPOCHS} - {time.time()-start:.0f}s - "
          f"loss: {train_loss:.4f} acc: {train_acc:.4f} - val_loss: {val_loss:.4f} val_acc: {val_acc:.4f}")

Epoch 1/5 - 388s - loss: 0.2163 acc: 0.9256 - val_loss: 0.0760 val_acc: 0.9727
Epoch 2/5 - 382s - loss: 0.0329 acc: 0.9893 - val_loss: 0.0390 val_acc: 0.9818
Epoch 3/5 - 382s - loss: 0.0158 acc: 0.9945 - val_loss: 0.0263 val_acc: 0.9914
Epoch 4/5 - 382s - loss: 0.0138 acc: 0.9963 - val_loss: 0.0170 val_acc: 0.9941
Epoch 5/5 - 381s - loss: 0.0077 acc: 0.9977 - val_loss: 0.0038 val_acc: 0.9984


**Cell 7 — Test evaluate:**



In [34]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(1).cpu().numpy()
        y_pred.extend(preds)
        y_true.extend(labels.numpy())

print(classification_report(y_true, y_pred, target_names=class_names))
print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

        Cyst       1.00      1.00      1.00       557
      Normal       1.00      1.00      1.00       763
       Stone       1.00      1.00      1.00       208
       Tumor       1.00      1.00      1.00       343

    accuracy                           1.00      1871
   macro avg       1.00      1.00      1.00      1871
weighted avg       1.00      1.00      1.00      1871

[[557   0   0   0]
 [  0 763   0   0]
 [  0   0 208   0]
 [  0   0   0 343]]


In [36]:
import os

for split in ['train', 'val', 'test']:
    total = 0
    for cls in class_names:
        cls_path = f'/content/split_data/{split}/{cls}'
        total += len(os.listdir(cls_path))
    print(split, "total images:", total)

train total images: 8710
val total images: 1865
test total images: 1871


In [37]:
import hashlib

def file_hash(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

train_hashes = {file_hash(f): f for f, _ in train_data.samples}
test_hashes = {file_hash(f): f for f, _ in test_data.samples}
val_hashes = {file_hash(f): f for f, _ in val_data.samples}

train_test_dupes = set(train_hashes) & set(test_hashes)
train_val_dupes = set(train_hashes) & set(val_hashes)

print("Exact duplicate images (Train-Test):", len(train_test_dupes))
print("Exact duplicate images (Train-Val):", len(train_val_dupes))

# Show a few examples if found
for h in list(train_test_dupes)[:5]:
    print("Train:", train_hashes[h])
    print("Test :", test_hashes[h])
    print()

Exact duplicate images (Train-Test): 111
Exact duplicate images (Train-Val): 94
Train: /content/split_data/train/Cyst/Cyst- (474).jpg
Test : /content/split_data/test/Cyst/Cyst- (394).jpg

Train: /content/split_data/train/Cyst/Cyst- (342).jpg
Test : /content/split_data/test/Cyst/Cyst- (422).jpg

Train: /content/split_data/train/Cyst/Cyst- (2305).jpg
Test : /content/split_data/test/Cyst/Cyst- (2343).jpg

Train: /content/split_data/train/Normal/Normal- (1594).jpg
Test : /content/split_data/test/Normal/Normal- (1669).jpg

Train: /content/split_data/train/Cyst/Cyst- (1258).jpg
Test : /content/split_data/test/Cyst/Cyst- (1108).jpg



In [38]:
import hashlib, os, shutil

def file_hash(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

# Hash every image across the whole (pre-split) dataset
seen_hashes = {}
duplicates = []

for cls in class_names:
    cls_path = os.path.join(DATA_PATH, cls)
    for fname in os.listdir(cls_path):
        fpath = os.path.join(cls_path, fname)
        h = file_hash(fpath)
        if h in seen_hashes:
            duplicates.append(fpath)
        else:
            seen_hashes[h] = fpath

print(f"Total duplicate images found: {len(duplicates)}")
print(f"Total unique images: {len(seen_hashes)}")

# Build a deduplicated copy of the dataset
DEDUP_PATH = '/content/data_dedup'
os.makedirs(DEDUP_PATH, exist_ok=True)

for cls in class_names:
    os.makedirs(os.path.join(DEDUP_PATH, cls), exist_ok=True)

for h, fpath in seen_hashes.items():
    cls = os.path.basename(os.path.dirname(fpath))
    fname = os.path.basename(fpath)
    shutil.copy(fpath, os.path.join(DEDUP_PATH, cls, fname))

print("Deduplicated dataset created at:", DEDUP_PATH)
for cls in class_names:
    print(cls, "->", len(os.listdir(os.path.join(DEDUP_PATH, cls))))

Total duplicate images found: 517
Total unique images: 11929
Deduplicated dataset created at: /content/data_dedup
Cyst -> 3284
Normal -> 5002
Stone -> 1360
Tumor -> 2283


In [35]:
torch.save(model.state_dict(), '/content/drive/MyDrive/Thesis /Final /vit_b16_kidney.pth')
print("Model saved!")

Model saved!
